In [11]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.ipynb) \|\|
[Quickstart](Quickstart.ipynb) \|\| **Tensors** \|\| [Datasets &
DataLoaders](data_tutorial.ipynb) \|\|
[Transforms](transforms_tutorial.ipynb) \|\| [Build
Model](buildmodel_tutorial.ipynb) \|\|
[Autograd](autogradqs_tutorial.ipynb) \|\|
[Optimization](optimization_tutorial.ipynb) \|\| [Save & Load
Model](saveloadrun_tutorial.ipynb)

Tensors
=======

Tensors are a specialized data structure that are very similar to arrays
and matrices. In PyTorch, we use tensors to encode the inputs and
outputs of a model, as well as the model's parameters.

Tensors are similar to [NumPy's](https://numpy.org/) ndarrays, except
that tensors can run on GPUs or other hardware accelerators. In fact,
tensors and NumPy arrays can often share the same underlying memory,
eliminating the need to copy data (see
`bridge-to-np-label`{.interpreted-text role="ref"}). Tensors are also
optimized for automatic differentiation (we\'ll see more about that
later in the [Autograd](autogradqs_tutorial.ipynb) section). If you're
familiar with ndarrays, you'll be right at home with the Tensor API. If
not, follow along!


In [12]:
import torch
import numpy as np

Initializing a Tensor
=====================

Tensors can be initialized in various ways. Take a look at the following
examples:

**Directly from data**

Tensors can be created directly from data. The data type is
automatically inferred.


In [13]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)
print(x_data)
print(x_data.dtype)

tensor([[1, 2],
        [3, 4]])
torch.int64


**From a NumPy array**

Tensors can be created from NumPy arrays (and vice versa - see
`bridge-to-np-label`{.interpreted-text role="ref"}).


In [14]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(x_np)


tensor([[1, 2],
        [3, 4]])


**From another tensor:**

The new tensor retains the properties (shape, datatype) of the argument
tensor, unless explicitly overridden.


In [15]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")
print(f"Dtype of tensor: {x_rand.dtype} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.8932, 0.9521],
        [0.7058, 0.1443]]) 

Dtype of tensor: torch.float32 



**With random or constant values:**

`shape` is a tuple of tensor dimensions. In the functions below, it
determines the dimensionality of the output tensor.


In [16]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.5428, 0.4456, 0.3819],
        [0.8494, 0.7532, 0.2664]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


------------------------------------------------------------------------


Attributes of a Tensor
======================

Tensor attributes describe their shape, datatype, and the device on
which they are stored.


In [17]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")
print(tensor.size())

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu
torch.Size([3, 4])


In [18]:
tensor.requires_grad

False

A PyTorch tensor stores the following:

**Data**
- The actual numerical values, held in a contiguous memory block called a `Storage` object (shared across views)

**Metadata**
| Attribute | Description | Example |
|---|---|---|
| `dtype` | Element data type | `torch.float32`, `torch.int64` |
| `shape` / `size()` | Dimensions of the tensor | `torch.Size([3, 4])` |
| `stride()` | Step size per dimension in storage | `(4, 1)` for a contiguous 3×4 tensor |
| `storage_offset()` | Offset into storage where this tensor's data starts | `0` for a fresh tensor, non-zero for slices |
| `device` | Where the data lives | `cpu`, `cuda:0` |
| `requires_grad` | Whether autograd tracks operations on it | `True` / `False` |
| `grad` | Accumulated gradient (after `.backward()`) | another tensor or `None` |
| `grad_fn` | The autograd function that created it (for non-leaf tensors) | `<AddBackward0>` or `None` |
| `is_leaf` | Whether it's a leaf node in the autograd graph | `True` / `False` |
| `layout` | Memory layout | `torch.strided` (default), `torch.sparse_coo` |
| `is_contiguous()` | Whether strides match row-major layout | `True` / `False` |

**Key insight — views share storage:**


In [19]:
x = torch.randn(3, 4)
y = x[1:]          # y is a view: same storage, different offset/shape/strides
x.data_ptr() != y.data_ptr()          # different start pointer
x.untyped_storage().data_ptr() == y.untyped_storage().data_ptr()  # but same storage

True

`stride()` describes **how many elements to skip in memory to move one step along each dimension**.

### Mental Model

Tensor data is stored as a **flat 1D array** in memory. Strides tell PyTorch how to translate a multi-dimensional index into a flat memory position:



flat_index = sum(index[i] * stride[i] for each dim i) + storage_offset



### Concrete Example



In [20]:
x = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])   # shape (2, 3)

x.stride()  # → (3, 1)

(3, 1)



Memory layout: `[1, 2, 3, 4, 5, 6]`

- To move **down one row** (dim 0): skip **3** elements → stride[0] = 3
- To move **right one column** (dim 1): skip **1** element → stride[1] = 1

So `x[1][2]` = element at `1*3 + 2*1 = 5` → value `6` ✓

### Why It Matters

Strides are what make zero-copy operations possible:

| Operation | Shape | Stride | Data Copied? |
|---|---|---|---|
| `x` (original 2×3) | `(2, 3)` | `(3, 1)` | — |
| `x.t()` (transpose) | `(3, 2)` | `(1, 3)` | ❌ No |
| `x[1:]` (slice) | `(1, 3)` | `(3, 1)` | ❌ No |
| `x.unsqueeze(0)` | `(1, 2, 3)` | `(6, 3, 1)` | ❌ No |

`x.t()` just **swaps the strides** — same storage, no data movement.

### Non-Contiguous Tensors

After a transpose, strides no longer follow row-major order — the tensor is **non-contiguous**:



In [21]:
x = torch.randn(2, 3)
x_t = x.t()

x.is_contiguous()    # True  — strides: (3, 1)
x_t.is_contiguous()  # False — strides: (1, 2)

False



This is exactly why `view()` fails on `x_t` — it requires contiguous strides to safely reinterpret the flat memory as a new shape.



The separation of **storage** (raw data) from **metadata** (shape, stride, offset) is what makes `view()`, `transpose()`, `unsqueeze()`, and slicing zero-copy operations.

## How Metadata and Data Change Across Operations

Helper to print all key metadata at once:

In [22]:
import torch

def tensor_info(name, t):
    print(f"{'─'*45}")
    print(f"  {name}")
    print(f"{'─'*45}")
    print(f"  shape          : {t.shape}")
    print(f"  stride         : {t.stride()}")
    print(f"  storage_offset : {t.storage_offset()}")
    print(f"  dtype          : {t.dtype}")
    print(f"  device         : {t.device}")
    print(f"  is_contiguous  : {t.is_contiguous()}")
    print(f"  storage ptr    : {t.untyped_storage().data_ptr()}")
    print()

base = torch.arange(1, 13, dtype=torch.float32).reshape(3, 4)
tensor_info("base  (3×4)", base)

─────────────────────────────────────────────
  base  (3×4)
─────────────────────────────────────────────
  shape          : torch.Size([3, 4])
  stride         : (4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768



### `view()` — Same Storage, New Shape & Stride

In [23]:
v = base.view(2, 6)
tensor_info("base         (3×4)", base)
tensor_info("base.view(2,6)", v)

print(f"Same storage? {base.untyped_storage().data_ptr() == v.untyped_storage().data_ptr()}")
# shape changes: (3,4) → (2,6)
# stride changes: (4,1) → (6,1)  — recalculated for the new shape
# storage_offset: unchanged (0)
# contiguous: still True
# storage ptr: identical — no data copied

─────────────────────────────────────────────
  base         (3×4)
─────────────────────────────────────────────
  shape          : torch.Size([3, 4])
  stride         : (4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

─────────────────────────────────────────────
  base.view(2,6)
─────────────────────────────────────────────
  shape          : torch.Size([2, 6])
  stride         : (6, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

Same storage? True


### `transpose()` / `.t()` — Strides Swap, Tensor Becomes Non-Contiguous

In [24]:
t = base.t()   # equivalent to base.transpose(0, 1)
tensor_info("base      (3×4)", base)
tensor_info("base.t()  (4×3)", t)

print(f"Same storage? {base.untyped_storage().data_ptr() == t.untyped_storage().data_ptr()}")
# shape changes:  (3,4) → (4,3)
# stride changes: (4,1) → (1,4)  — strides are simply swapped
# is_contiguous:  True  → False   — memory order no longer matches row-major
# storage ptr:    identical — no data copied

# Because t is non-contiguous, view() will raise RuntimeError:
try:
    t.view(12)
except RuntimeError as e:
    print(f"\nview() on transposed tensor → RuntimeError: {e}")

─────────────────────────────────────────────
  base      (3×4)
─────────────────────────────────────────────
  shape          : torch.Size([3, 4])
  stride         : (4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

─────────────────────────────────────────────
  base.t()  (4×3)
─────────────────────────────────────────────
  shape          : torch.Size([4, 3])
  stride         : (1, 4)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : False
  storage ptr    : 5643856768

Same storage? True

view() on transposed tensor → RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


### `unsqueeze()` — New Size-1 Dim Inserted, Stride Gets a New Entry

In [25]:
u0 = base.unsqueeze(0)   # insert dim at position 0
u1 = base.unsqueeze(1)   # insert dim at position 1
u2 = base.unsqueeze(2)   # insert dim at position 2

tensor_info("base          (3×4)",     base)
tensor_info("unsqueeze(0)  (1×3×4)",   u0)
tensor_info("unsqueeze(1)  (3×1×4)",   u1)
tensor_info("unsqueeze(2)  (3×4×1)",   u2)

print(f"All share same storage as base?")
for name, u in [("unsqueeze(0)", u0), ("unsqueeze(1)", u1), ("unsqueeze(2)", u2)]:
    same = base.untyped_storage().data_ptr() == u.untyped_storage().data_ptr()
    print(f"  {name}: {same}")
# shape:  a new size-1 dimension is inserted at the given position
# stride: a corresponding stride entry is inserted; its value equals the product
#         of sizes of all dims that come after it (keeps addressing consistent)
# storage_offset: unchanged
# is_contiguous: remains True
# storage ptr: identical — pure metadata change, zero data copied

─────────────────────────────────────────────
  base          (3×4)
─────────────────────────────────────────────
  shape          : torch.Size([3, 4])
  stride         : (4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

─────────────────────────────────────────────
  unsqueeze(0)  (1×3×4)
─────────────────────────────────────────────
  shape          : torch.Size([1, 3, 4])
  stride         : (12, 4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

─────────────────────────────────────────────
  unsqueeze(1)  (3×1×4)
─────────────────────────────────────────────
  shape          : torch.Size([3, 1, 4])
  stride         : (4, 4, 1)
  storage_offset : 0
  dtype          : torch.float32
  device         : cpu
  is_contiguous  : True
  storage ptr    : 5643856768

─────────────────────────────────────────────
  unsqu

### Summary: What Changes vs What Stays the Same

| Operation | `shape` | `stride` | `storage_offset` | `is_contiguous` | Storage (data) |
|---|---|---|---|---|---|
| `view(new_shape)` | ✅ changes | ✅ recalculated | unchanged | ✅ True | **shared** |
| `t()` / `transpose()` | ✅ dims swap | ✅ swapped | unchanged | ❌ False | **shared** |
| `unsqueeze(dim)` | ✅ +1 dim (size 1) | ✅ +1 entry | unchanged | ✅ True | **shared** |
| `clone()` | same | same | same | same | **copied** |

All three zero-copy operations only modify metadata — the underlying storage is never touched.

### Stride Intuition: "Size of One Slice"

> **stride[i] = how many elements to skip to step once along dimension i**
> = the total size of everything that lives *inside* that dimension
> = product of all shape sizes that come **after** dimension i

**2D — shape `(3, 4)` → stride `(4, 1)`:**

```
Row 0: [ 0,  1,  2,  3 ]
Row 1: [ 4,  5,  6,  7 ]   ← stride[0]=4: "a row is 4 wide, so jump 4 to reach the next row"
Row 2: [ 8,  9, 10, 11 ]
                  ↑ stride[1]=1: "move one element to go right"
```

**3D — shape `(2, 3, 4)` → stride `(12, 4, 1)`:**

| Dim | Meaning | Stride = ? | Why |
|-----|---------|-----------|-----|
| 0 (batch) | move to next matrix | **12** | one matrix = 3×4 = 12 elements |
| 1 (row)   | move to next row    | **4**  | one row = 4 elements |
| 2 (col)   | move to next element| **1**  | last dim always 1 |

**`unsqueeze` edge case — shape `(3, 1, 4)` → stride `(4, 4, 1)`:**

The inserted size-1 dimension gets stride = product of what follows it (`4×1 = 4`). Since you can only ever be at index 0 of a size-1 dim, the value doesn't affect addressing — it's just kept consistent.

------------------------------------------------------------------------


Operations on Tensors
=====================

Over 1200 tensor operations, including arithmetic, linear algebra,
matrix manipulation (transposing, indexing, slicing), sampling and more
are comprehensively described
[here](https://pytorch.org/docs/stable/torch.html).

Each of these operations can be run on the CPU and
[Accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If you're using Colab, allocate an
accelerator by going to Runtime \> Change runtime type \> GPU.

By default, tensors are created on the `CPU`. We need to explicitly move
tensors to the accelerator using `.to` method (after checking for
accelerator availability). Keep in mind that copying large tensors
across devices can be expensive in terms of time and memory!


In [36]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())
print(f"Tensor is stored on: {tensor.device}")

Tensor is stored on: mps:0


Try out some of the operations from the list. If you\'re familiar with
the NumPy API, you\'ll find the Tensor API a breeze to use.


**Standard numpy-like indexing and slicing:**


In [40]:
tensor = torch.randint(0, 10, (4, 4))
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([5, 4, 4, 4])
First column: tensor([5, 0, 9, 1])
Last column: tensor([4, 5, 6, 3])
tensor([[5, 0, 4, 4],
        [0, 0, 7, 5],
        [9, 0, 8, 6],
        [1, 0, 6, 3]])


**Joining tensors** You can use `torch.cat` to concatenate a sequence of
tensors along a given dimension. See also
[torch.stack](https://pytorch.org/docs/stable/generated/torch.stack.html),
another tensor joining operator that is subtly different from
`torch.cat`.


In [41]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[5, 0, 4, 4, 5, 0, 4, 4, 5, 0, 4, 4],
        [0, 0, 7, 5, 0, 0, 7, 5, 0, 0, 7, 5],
        [9, 0, 8, 6, 9, 0, 8, 6, 9, 0, 8, 6],
        [1, 0, 6, 3, 1, 0, 6, 3, 1, 0, 6, 3]])


**Arithmetic operations**


In [45]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

# --- Matrix multiplication (row dot column) ---
# y1, y2, y3 all produce the same result — three equivalent syntaxes
y1 = a @ b                          # operator overload
y2 = a.matmul(b)                    # method call
y3 = torch.empty(2, 2)
torch.matmul(a, b, out=y3)          # function with out= (writes into pre-allocated buffer)

print("Matrix multiplication (a @ b):")
print(y1)
print(f"y1 == y2: {y1.equal(y2)},  y1 == y3: {y1.equal(y3)}")

# --- Element-wise multiplication ---
# z1, z2, z3 all produce the same result
z1 = a * b                          # operator overload
z2 = a.mul(b)                       # method call
z3 = torch.empty(2, 2)
torch.mul(a, b, out=z3)             # function with out=

print("\nElement-wise multiplication (a * b):")
print(z1)
print(f"z1 == z2: {z1.equal(z2)},  z1 == z3: {z1.equal(z3)}")

Matrix multiplication (a @ b):
tensor([[19., 22.],
        [43., 50.]])
y1 == y2: True,  y1 == y3: True

Element-wise multiplication (a * b):
tensor([[ 5., 12.],
        [21., 32.]])
z1 == z2: True,  z1 == z3: True


**Single-element tensors** If you have a one-element tensor, for example
by aggregating all values of a tensor into one value, you can convert it
to a Python numerical value using `item()`:


In [46]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

16.0 <class 'float'>


**In-place operations** Operations that store the result into the
operand are called in-place. They are denoted by a `_` suffix. For
example: `x.copy_(y)`, `x.t_()`, will change `x`.


In [47]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]) 

tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>In-place operations save some memory, but can be problematic when computing derivatives because of an immediate lossof history. Hence, their use is discouraged.</p>

</div>



------------------------------------------------------------------------


Bridge with NumPy {#bridge-to-np-label}
=================

Tensors on the CPU and NumPy arrays can share their underlying memory
locations, and changing one will change the other.


Tensor to NumPy array
=====================


In [48]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


A change in the tensor reflects in the NumPy array.


In [49]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


NumPy array to Tensor
=====================


In [50]:
n = np.ones(5)
t = torch.from_numpy(n)

Changes in the NumPy array reflects in the tensor.


In [51]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]
